# 03 · mPLUG-DocOwl 2

**mPLUG-DocOwl 2** — модель Alibaba для понимания документов с shape-adaptive cropping. Чекпоинт: [`mPLUG/DocOwl2`](https://huggingface.co/mPLUG/DocOwl2).

Ноутбук берёт subset, сохранённый в `data/subset.json` ноутбуком `01_setup_and_dataset.ipynb`, и прогоняет на нём модель. Результаты записываются в `results/<model>/predictions.jsonl`.

## 1. Установка

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# === Colab / Kaggle bootstrap =================================================
# В Colab клонируем репозиторий проекта (предполагается, что код выложен
# в GitHub) и переходим в его корень. Для локального запуска просто
# проверьте, что текущая рабочая директория — корень ocr_eval/.
import os, sys, subprocess, pathlib

REPO_NAME = "ocr_eval"
if not pathlib.Path(REPO_NAME).exists():
    # Замените URL на ваш форк, если работаете в Colab
    # !git clone https://github.com/<your-username>/ocr_eval.git
    pass

if pathlib.Path(REPO_NAME).exists():
    os.chdir(REPO_NAME)
sys.path.insert(0, str(pathlib.Path.cwd() / "src"))
print("CWD =", os.getcwd())


## 2. Конфиг и загрузка модели

In [ ]:
from src.utils import load_config, JsonlWriter, Timer, cuda_free, gpu_info, already_processed_ids
from src.io_records import PredictionRecord

cfg = load_config('configs/mplug_docowl.yaml')
cfg

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_REPO = cfg['model']['hf_repo']
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_REPO,
    trust_remote_code=True,
    torch_dtype=getattr(torch, cfg['model']['torch_dtype']),
    device_map=cfg['model']['device_map'],
).eval()
print(gpu_info())

## 3. Subset и helper

In [ ]:
import json, pathlib
from src.dataset_loader import GroundTruth

DATA_ROOT = pathlib.Path('data/OmniDocBench')
subset = [GroundTruth(**rec) for rec in json.loads(
    pathlib.Path('data/subset.json').read_text(encoding='utf-8'))]
print(f'subset: {len(subset)} страниц')

In [ ]:
# DocOwl ожидает PIL.Image + текстовый промпт; в зависимости от версии
# либо есть метод model.chat(images=[img], query=prompt), либо нужно
# собрать messages вручную. Универсальный путь — через preprocessor:
from transformers import AutoProcessor
try:
    processor = AutoProcessor.from_pretrained(MODEL_REPO, trust_remote_code=True)
except Exception:
    processor = None
    print('processor не нужен — используем model.chat() напрямую')


## 4. Инференс

In [ ]:
import traceback
from pathlib import Path
from PIL import Image

out_path = Path(cfg['output']['results_dir']) / 'predictions.jsonl'
out_path.parent.mkdir(parents=True, exist_ok=True)
done = already_processed_ids(out_path)

PROMPT = cfg['inference']['prompt']
MAX_NEW = cfg['inference']['max_new_tokens']

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done: continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists(): continue
        rec = PredictionRecord(page_id=gt.page_id, model='mplug_docowl')
        try:
            img = Image.open(img_path).convert('RGB')
            with Timer('infer') as t:
                if hasattr(model, 'chat'):
                    out = model.chat(image=img, msgs=[{'role':'user','content': PROMPT}],
                                     tokenizer=tokenizer, sampling=False, max_new_tokens=MAX_NEW)
                else:
                    inputs = processor(images=img, text=PROMPT, return_tensors='pt').to(model.device)
                    with torch.no_grad():
                        gen = model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=False)
                    out = processor.batch_decode(gen, skip_special_tokens=True)[0]
            rec.full_text = out if isinstance(out, str) else str(out)
            rec.raw_output = rec.full_text
            rec.inference_time_s = t.elapsed
        except Exception as e:
            rec.error = f'{type(e).__name__}: {e}'
            traceback.print_exc()
        w.write(rec.to_dict())
print('готово →', out_path)

## 5. Освободить GPU

In [ ]:
del model; cuda_free(); print(gpu_info())